# How to connect to a Lakehouse via the SQL Analytics Endpoint using the workspace identity

I've needed this for the occasional INFORMATION_SCHEMA queries since it appears that spark has issues interrogating the structure of a lakehouse that has schemas enabled for tables. 

In [ ]:
def get_lakehouse_tables_sqlalchemy_engine(endpoint_host: str, database: str) -> Engine:
    """
    Get a SQLAlchemy engine for the BeODS Fabric database.

    Returns:
        sqlalchemy.engine.Engine: The SQLAlchemy engine object.
    """
    import struct
    import pyodbc
    from sqlalchemy import create_engine
    from sqlalchemy.engine import URL
    
    # Get and prepare the token
    token = notebookutils.credentials.getToken("pbi")
    tokenb = bytes(token, "UTF-8")
    exptoken = b""

    for i in tokenb:
        exptoken += bytes({i})
        exptoken += bytes(1)

    tokenstruct = struct.pack("=i", len(exptoken)) + exptoken
   
    
    # Create SQLAlchemy connection URL
    connection_url = URL.create(
        "mssql+pyodbc",
        host=endpoing_host,
        database=database,
        query={
            "driver": "ODBC Driver 18 for SQL Server",
            "TrustServerCertificate": "yes",
            "Encrypt": "yes"
        }
    )
    
    # PyODBC can use handle the token via attrs_before
    def creator():
        SQL_COPY_SS_ACCESS_TOKEN = 1256
        driver = "Driver={ODBC Driver 18 for SQL Server}"
        server = f";SERVER={host}"
        database_conn = f";DATABASE={database}"
        connString = driver + server + database_conn
        
        return pyodbc.connect(
            connString, 
            attrs_before={SQL_COPY_SS_ACCESS_TOKEN: tokenstruct}
        )
    
    # Create the SQLAlchemy engine with the custom creator
    engine = create_engine(
        connection_url,
        creator=creator,
        echo=False  # Set to True for SQL debugging
    )    
    return engine